In [1]:
%pip install uv --quiet
!uv pip install pandas numpy plotly matplotlib
!uv sync

Note: you may need to restart the kernel to use updated packages.


Using Python 3.11.15 environment at: c:\Users\logan\OneDrive\Documents\SeniorSpring\Adv Data Sci\Modern-Store-Of-Value\.venv
Checked 4 packages in 11ms
Resolved 132 packages in 2ms
Checked 128 packages in 13ms


In [2]:
# Data manipulation tools
import pandas as pd
import datetime

# Visualization tools
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

In [3]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust (Replaces BTC-USD)
        "ETHA"   # iShares Ethereum Trust (Replaces ETH-USD)
    ], 
    
    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],
    
    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],
    
    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI"   # Total US Market (replaces Wilshire 5000)
    ],
    
    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],
    
    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [4]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    # Stores valid tickers
    valid_tickers = [
        t for t in flat_tickers
        if processor.has_ticker(t)
    ]

    # Prints tickers that were not valid
    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

# data['AAPL'].tail()
# data["BITW"]
# combined_data[combined_data['Ticker'] == 'BITW']
print(combined_data)
# combined_data

           Date     Open     High      Low    Close       Volume  OpenInt  \
0    2025-12-31  62.8200  63.1200  56.5600  58.7600    1752680.0        0   
1    2026-01-31  59.9400  66.4800  54.5000  55.6601    2425192.0        0   
2    2026-02-28  51.2878  52.2950  40.6599  43.0400    4596906.0        0   
3    2026-03-26  42.9952  49.4500  42.9952  44.8700    1583267.0        0   
4    2024-01-31  26.4000  26.4100  22.0200  24.3000  207876989.0        0   
...         ...      ...      ...      ...      ...          ...      ...   
1307 2025-11-30  26.6000  26.9400  25.5500  26.4200    3917732.0        0   
1308 2025-12-31  26.3600  26.6666  25.4000  25.5200    4877035.0        0   
1309 2026-01-31  25.5000  26.0500  25.4250  25.6600    5404587.0        0   
1310 2026-02-28  25.5500  26.1500  25.5400  26.0200    5422359.0        0   
1311 2026-03-26  26.0400  27.1429  25.8600  27.1100   37145895.0        0   

     Ticker                      Category  
0      BITW                   C

## Metric - Maximum Downturn

In [5]:
"""
Context: 
- Maximum Downturn: Peak-to-trough decline during specific periods to assess downside risk
- In simple terms, this metric measures the worst-case scenario: the largest percentage 
  drop an asset experienced from its highest peak to its lowest subsequent trough.
- The lower the percentage, the better "safe haven" qualities an asset has, 
  as it indicates less severe losses during market downturns.
"""

'\nContext: \n- Maximum Downturn: Peak-to-trough decline during specific periods to assess downside risk\n- In simple terms, this metric measures the worst-case scenario: the largest percentage \n  drop an asset experienced from its highest peak to its lowest subsequent trough.\n- The lower the percentage, the better "safe haven" qualities an asset has, \n  as it indicates less severe losses during market downturns.\n'

In [6]:
# Sort by ticker and date
combined_data = combined_data.sort_values(by=['Ticker', 'Date'])

# Calculate rolling peak / maximum with groupby and cummax
combined_data['Peak'] = combined_data.groupby('Ticker')['Close'].cummax()

# Calculate percentage drawdowns from peak
combined_data['Drawdown'] = (combined_data['Close'] - combined_data['Peak']) / combined_data['Peak']

# Find lowest drawdown value
max_downturn_df = (
    combined_data.groupby(['Category', 'Ticker'])['Drawdown']
    .min()
    .reset_index()
    .rename(columns={'Drawdown': 'Max_Downturn'})
)

# Sort from worst to best
max_downturn_df = max_downturn_df.sort_values(by='Max_Downturn')

# Formatting
max_downturn_df['Max_Downturn_Formatted'] = (max_downturn_df['Max_Downturn'] * 100).round(2).astype(str) + '%'

# Print results
print(max_downturn_df)

                        Category Ticker  Max_Downturn Max_Downturn_Formatted
6        Commodity ETFs (Metals)   PALL     -0.695893                -69.59%
20             Individual Stocks   TSLA     -0.677190                -67.72%
4   Commodity ETFs (Agriculture)   WEAT     -0.635916                -63.59%
19             Individual Stocks   NVDA     -0.628203                -62.82%
13             Individual Stocks    AMD     -0.620762                -62.08%
10                   Crypto ETFs   ETHA     -0.557587                -55.76%
14             Individual Stocks   AMZN     -0.520969                 -52.1%
11                   Crypto ETFs   IBIT     -0.439234                -43.92%
8        Commodity ETFs (Metals)    SLV     -0.359846                -35.98%
15             Individual Stocks     HD     -0.330938                -33.09%
17             Individual Stocks    LOW     -0.319266                -31.93%
18             Individual Stocks   MSFT     -0.314021                 -31.4%

In [7]:
"""
BASELINE: 
- GOLD: -17.62%. With that being said we would like something to be 17% or better

WINNERS: 
- DBA (Broad Agriculture): At -10.15%, it had the lowest downturn of any asset. It acted as an incredibly stable store of value.

- XLU (Utilities): At -17.63%, it practically perfectly matches Gold's downside protection. This completely validates the finding mentioned in your report that XLU possesses strong defensive characteristics.

- JNJ & WMT (Consumer Staples): With drawdowns of -18.77% and -20.24%, these defensive stocks outperformed the broader market (SPY at -23.92%) and proved highly resilient during downturns.

LOSERS: 
- Palladium (PALL), Tesla (TSLA), Wheat (WEAT), Nvidia (NVDA), and AMD
  - All of these assets lost over 60% of their value from their peak at some point. 
  - Even though assets like NVDA have massive long-term returns, a 62% drawdown means an investor could lose 
    more than half their wealth in a crisis. They are speculative growth assets, not stores of value.
    
CRYPTOCURRENCIES:
- ETHA (Ethereum) and IBIT (Bitcoin): 
  - Experienced severe crashes of -55.76% and -43.92%. They failed the stability test and behave much more 
    like high-growth tech stocks than "digital gold."

- BITW (Crypto Index): 
  - The diversification of the index helped soften the blow to -26.75%, but it still suffered worse drawdowns
    than the broader stock market (SPY/VTI). Crypto broadly failed to preserve capital during market corrections.
"""


'\nBASELINE: \n- GOLD: -17.62%. With that being said we would like something to be 17% or better\n\nWINNERS: \n- DBA (Broad Agriculture): At -10.15%, it had the lowest downturn of any asset. It acted as an incredibly stable store of value.\n\n- XLU (Utilities): At -17.63%, it practically perfectly matches Gold\'s downside protection. This completely validates the finding mentioned in your report that XLU possesses strong defensive characteristics.\n\n- JNJ & WMT (Consumer Staples): With drawdowns of -18.77% and -20.24%, these defensive stocks outperformed the broader market (SPY at -23.92%) and proved highly resilient during downturns.\n\nLOSERS: \n- Palladium (PALL), Tesla (TSLA), Wheat (WEAT), Nvidia (NVDA), and AMD\n  - All of these assets lost over 60% of their value from their peak at some point. \n  - Even though assets like NVDA have massive long-term returns, a 62% drawdown means an investor could lose \n    more than half their wealth in a crisis. They are speculative growth a

## Inflation Adjusted Returns For All

In [ ]:
import pandas_datareader.data as web

# Fetch FRED data
cpi_data = web.DataReader('CPIAUCSL', 'fred', start_date, end_date)

# Calculate  total percentage increase inflation
cpi_start_val = cpi_data['CPIAUCSL'].iloc[0]
cpi_end_val = cpi_data['CPIAUCSL'].iloc[-1]
total_inflation_pct = ((cpi_end_val - cpi_start_val) / cpi_start_val) * 100

print(f"Total CPI Inflation ({start_date} to {end_date}): {total_inflation_pct:.2f}%\n")

# Ensure datetime
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Monthly data, take last price of each month
monthly_data = (
    combined_data.set_index('Date')
    .groupby(['Ticker', 'Category'])
    .resample('MS')['Close']
    .last()
    .reset_index()
)

# Store results
results = []

# For each ticker, calculate start price, end price, total return, and inflation-adjusted return
for ticker in monthly_data['Ticker'].unique():
    # Grab specific asset
    ticker_df = monthly_data[monthly_data['Ticker'] == ticker].sort_values('Date')
    
    # If no data, skip
    if ticker_df.empty:
        continue
        
    # Calculate start and end price
    start_price = ticker_df.iloc[0]['Close']
    end_price = ticker_df.iloc[-1]['Close']
    
    # Calculate asset's overall return
    asset_return_pct = ((end_price - start_price) / start_price) * 100
    
    # Calculate real return
    real_return_pct = asset_return_pct - total_inflation_pct
    
    # Append results
    results.append({
        'Category': ticker_df.iloc[0]['Category'],
        'Ticker': ticker,
        'Total_Return_%': asset_return_pct,
        'Total_Inflation_%': total_inflation_pct,
        'Inflation_Adjusted_Return_%': real_return_pct
    })

# Display results in dataframe
returns_df = pd.DataFrame(results)

# Sort best to worst performing
returns_df = returns_df.sort_values(by='Inflation_Adjusted_Return_%', ascending=False)

# % formatting for clean reporting
formatted_df = returns_df.copy()
formatted_df['Total_Return_%'] = formatted_df['Total_Return_%'].round(2).astype(str) + '%'
formatted_df['Total_Inflation_%'] = formatted_df['Total_Inflation_%'].round(2).astype(str) + '%'
formatted_df['Inflation_Adjusted_Return_%'] = formatted_df['Inflation_Adjusted_Return_%'].round(2).astype(str) + '%'

# Display the data
display(formatted_df)

formatted_df.to_csv('../data/inflation_adjusted_returns.csv', index=False)

Total CPI Inflation (2021-01-01 to 2026-03-31): 24.66%



,Category,Ticker,Total_Return_%,Total_Inflation_%,Inflation_Adjusted_Return_%
12,Individual Stocks,NVDA,1221.63%,24.66%,1196.97%
21,Individual Stocks,WMT,173.32%,24.66%,148.66%
15,Commodity ETFs (Metals),SLV,143.18%,24.66%,118.52%
1,Individual Stocks,AMD,137.94%,24.66%,113.28%
6,Commodity ETFs (Metals),GLD,132.11%,24.66%,107.45%
0,Individual Stocks,AAPL,96.87%,24.66%,72.21%
17,Broad Market ETFs,SPY,85.14%,24.66%,60.48%
22,Sector ETFs,XLU,69.87%,24.66%,45.21%
19,Broad Market ETFs,VTI,68.81%,24.66%,44.15%
4,Commodity ETFs (Agriculture),DBA,65.41%,24.66%,40.75%
